# Omnipose on DIC image

Try omnipose 2d models on the DIC image to segment hypha - some flow fields were good, but need improved cell probability to constrain the cell region. May train a RF model to segment cell from bg.

Omnipose also has a 3d model (`plant_omni`) that does native 3d segmentation. Tried this on Cy5+DAPI 2 channel z stacks.

## Omnipose 2d on DIC

In [ ]:
# make local editable packages automatically reload
%load_ext autoreload
%autoreload 2

In [ ]:
import omnipose
from omnipose.gpu import use_gpu
use_GPU = use_gpu()
use_GPU

# set up plotting defaults
omnipose.plot.setup()

# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['figure.dpi'] = 300
plt.style.use('dark_background')
%matplotlib inline

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer

search_path = (
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/03",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/04",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/05",
)

file_pattern = ("CET*DAPI*.tif",
                "CET*Cy5*.tif",
                "*DIC*.tif",
                "MAX_CET145*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit",
        "correct_DIC_shift" : [0,0] # [Y,X] more negative Y to shift up, more positive X to shift to right
    }
}

dic_files = []
for p in search_path:
    dic_files.extend(find_files_by_pattern(p, file_pattern[2], verbose=True))
    
dic_imgs = [ImageContainer(img,config) for img in dic_files]
imgs = [i.merge() for i in dic_imgs]

In [ ]:
import numpy as np
import cv2

def invert_image_colors(image: np.ndarray) -> np.ndarray:
    """
    Performs a color inversion on the input image.

    This function inverts the pixel values of an image. For an 8-bit image,
    a pixel with value `p` will become `255 - p`. For a 16-bit image,
    a pixel with value `p` will become `65535 - p`, and so on.
    It works for both grayscale and multi-channel (e.g., RGB) images.
    
    For logical matrices (boolean arrays), 0 (False) becomes 1 (True) and 
    1 (True) becomes 0 (False).

    Args:
        image (np.ndarray): The input image matrix. Can be grayscale (2D) or color (3D).

    Returns:
        np.ndarray: The color-inverted image.
    """
    if not isinstance(image, np.ndarray):
        raise TypeError("Input 'image' must be a NumPy array.")

    # Handle logical matrices (boolean arrays)
    if image.dtype == bool:
        return np.logical_not(image)

    # cv2.bitwise_not performs a bitwise NOT operation on each pixel.
    # For an 8-bit unsigned integer (uint8), this effectively inverts the color.
    # E.g., 0 (binary 00000000) becomes 255 (binary 11111111),
    # and 255 becomes 0.
    inverted_image = cv2.bitwise_not(image)

    return inverted_image

In [ ]:
import matplotlib.pyplot as plt
import time
import os
import torch
from pathlib import Path
from cellpose_omni import models
from cellpose_omni.models import CP_MODELS, C2_MODELS, C2_BD_MODELS, C1_BD_MODELS, C1_MODELS

# Ensure environment variable is set
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Cellpose-trained vs Omnipose-trained model registries, taken directly from
# cellpose_omni/models.py + omnipose/core.py. Using the library's own model
# lists (rather than a naive `'omni' in model_name` check) matters because
# 'bact_phase_affinity' is an Omnipose-family model with no "omni" in its name.
CELLPOSE_MODEL_NAMES = set(CP_MODELS + C2_MODELS)
OMNIPOSE_MODEL_NAMES = set(C2_BD_MODELS + C1_BD_MODELS + C1_MODELS)

def is_omni_model(name: str) -> bool:
    if name in OMNIPOSE_MODEL_NAMES:
        return True
    if name in CELLPOSE_MODEL_NAMES:
        return False
    raise ValueError(f"Unknown model '{name}'; add it to CELLPOSE_MODEL_NAMES or OMNIPOSE_MODEL_NAMES.")

# Define models to iterate over
model_names = [
    'bact_phase_affinity',
    'bact_phase_omni',
    'bact_fluor_omni',
    'worm_omni',
    'worm_high_res_omni',
    'worm_bact_omni',
    'bact_phase_cp',
    'bact_fluor_cp',
    'cyto2_omni',
    'cyto2',
    'cyto'
]

# Define parameters
params = {
    'channels': None,
    'rescale': None,
    'mask_threshold': -2,
    'flow_threshold': 0,
    'transparency': True,
    'cluster': True,        # only takes effect when omni=True
    'resample': True,
    'verbose': False,
    'tile': True,
    'niter': None,
    'augment': False,
    'affinity_seg': False,  # only takes effect when omni=True
    'diameter': 25
}

# Prepare the image once
img = imgs[0]

# Initialize the figure for composite plotting
# Rows = models, Cols = 4 (Image+Mask, Flow, Cell Probability, Thresholded Cell Probability)
n_models = len(model_names)
fig, axes = plt.subplots(n_models, 4, figsize=(20, 4 * n_models), constrained_layout=True)

# Handle case of single model (axes would be 1D array)
if n_models == 1:
    axes = axes.reshape(1, -1)

print(f"Starting evaluation of {n_models} models...")

for i, model_name in enumerate(model_names):
    omni = is_omni_model(model_name)
    print(f"[{i+1}/{n_models}] Running model: {model_name} (omni={omni})")
    
    # Initialize model
    # Assuming 'use_GPU' is defined in your notebook environment
    model = models.CellposeModel(gpu=use_GPU, model_type=model_name)
    
    # Run segmentation
    tic = time.time()
    masks, flows, styles = model.eval(img, **dict(params, omni=omni))
    net_time = time.time() - tic
    print(f'  -> Segmentation time: {net_time:.2f}s')
    
    # Extract visualization components
    # flows[0] is the RGB flow image
    flow_rgb = flows[0]
    cell_prob = flows[2]
    
    # --- Plotting ---
    
    # 1. Image + Mask Overlay
    ax_img = axes[i, 0]
    ax_img.imshow(img, cmap='gray')
    if masks.max() > 0:
        # Overlay masks with transparency using a distinct colormap (e.g., nipy_spectral)
        ax_img.imshow(masks, cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
    ax_img.set_title(f"{model_name}\nImage + Mask")
    ax_img.axis('off')
    
    # 2. Flow Field
    ax_flow = axes[i, 1]
    ax_flow.imshow(flow_rgb)
    ax_flow.set_title("Flow Field")
    ax_flow.axis('off')
    
    # 3. Cell Probability
    ax_prob = axes[i, 2]
    ax_prob.imshow(cell_prob, cmap='gray')
    ax_prob.set_title("Cell Probability")
    ax_prob.axis('off')

    # 4. Thresholded Cell Probability (same threshold used in mask reconstruction)
    ax_thresh = axes[i, 3]
    ax_thresh.imshow(cell_prob > params['mask_threshold'], cmap='gray')
    ax_thresh.set_title(f"Thresholded Cell Probability\n(mask_threshold={params['mask_threshold']})")
    ax_thresh.axis('off')

    # Release this model's GPU memory before loading the next one, so sequentially
    # loading many different U-Net configs (varying nchan/nclasses) doesn't leave
    # fragmented/stale CUDA memory behind for later iterations.
    del model, masks, flows, styles
    torch.cuda.empty_cache()

# --- Save Output ---
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/omnipose_DIC")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = dic_files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

# Construct filename using the stem of the first file
output_filename = current_output_dir / f"{dic_files[0].stem}_omnipose_comparison.png"

plt.savefig(output_filename, dpi=300)
print(f"Saved comparison figure to: {output_filename}")
plt.close(fig)
# plt.show()

Conclusion: bact_phase_affinity, bact_phase_omni, and bact_fluor_cp are the most promising

## RF prediction of cell prob

The cell probabilities predicted by omnipose is not very accurate. We use the RF plugin to improve it.

In [ ]:
import tifffile
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path

import omnipose
from omnipose.gpu import use_gpu
use_GPU = use_gpu()

# set up plotting defaults
omnipose.plot.setup()

# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['figure.dpi'] = 300
plt.style.use('dark_background')
%matplotlib inline

img_path = Path('~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/03/CET155_Cocu_Phalloidin_01_DIC.tif')

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "16bit",
        "correct_DIC_shift" : [0,0] # [Y,X] more negative Y to shift up, more positive X to shift to right
    }
}

img = ImageContainer(img_path.expanduser(), config)

cell_prob_path = Path('~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/03/CET155_Cocu_Phalloidin_01_DIC/CET155_Cocu_Phalloidin_01_DIC_full_probs.tif')
cell_prob = tifffile.imread(str(cell_prob_path.expanduser()))
cell_prob = cell_prob[1,:]

In [ ]:
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation, generate_binary_structure, binary_fill_holes

dilation_radius = 8
iscell = cell_prob > 0.8
struct = generate_binary_structure(2, 1)  # cross-shaped element, works for 2D and 3D
iscell = binary_dilation(iscell, structure=struct, iterations=dilation_radius)
iscell = binary_fill_holes(iscell)

plt.imshow(iscell)

In [ ]:
import numpy as np
import omnipose.core
from scipy.ndimage import binary_dilation, generate_binary_structure, binary_fill_holes

def reconstruct_masks_with_rf_cellprob(
    flows,
    rf_cellprob,
    mask_threshold=0.5,
    dilation_radius=5,
    flow_threshold=0.4,
    min_size=15,
    max_size=None,
    niter=None,
    cluster=False,
    affinity_seg=False,
    boundary_seg=False,
    nclasses=3,
    dim=2,
    use_gpu=False,
    device=None,
    verbose=False,
    **kwargs
):
    dP = flows[1]
    bd = flows[4]

    if bd is None or (isinstance(bd, np.ndarray) and bd.size == 0):
        bd = np.zeros_like(rf_cellprob)

    iscell = rf_cellprob > mask_threshold

    if dilation_radius > 0:
        struct = generate_binary_structure(dim, 1)
        iscell = binary_dilation(iscell, structure=struct, iterations=dilation_radius)
    
    iscell = binary_fill_holes(iscell)

    masks, p, tr, bounds, affinity = omnipose.core.compute_masks(
        dP,
        rf_cellprob,
        bd=bd,
        iscell=iscell,
        niter=niter,
        mask_threshold=mask_threshold,
        flow_threshold=flow_threshold,
        min_size=min_size,
        max_size=max_size,
        cluster=cluster,
        affinity_seg=affinity_seg,
        boundary_seg=boundary_seg,
        nclasses=nclasses,
        dim=dim,
        use_gpu=use_gpu,
        device=device,
        verbose=verbose,
        **kwargs
    )

    return masks, bounds, p, tr, affinity, iscell

In [ ]:
import numpy as np
from cellpose_omni import models
import omnipose.core
import time

img_arr = img.merge()

model_names = ['bact_phase_affinity', 'bact_phase_omni', 'bact_fluor_cp']

eval_params = dict(
    channels=None,
    rescale=None,
    mask_threshold=0.0,   # not used for masking; only affects flow normalisation internally
    flow_threshold=0.0,
    omni=True,
    cluster=False,
    resample=True,
    tile=True,
    augment=False,
    affinity_seg=False,
    transparency=False,
    verbose=False,
    compute_masks=False,  # skip internal mask step; we use RF cellprob instead
)

results = {}

for model_name in model_names:
    print(f'Running {model_name}...')
    model = models.CellposeModel(gpu=use_GPU, model_type=model_name)

    tic = time.time()
    _, flows, _ = model.eval(img_arr, **eval_params)
    print(f'  network: {time.time()-tic:.1f}s')

    tic = time.time()
    masks, bounds, p, tr, affinity, iscell = reconstruct_masks_with_rf_cellprob(
        flows=flows,
        rf_cellprob=cell_prob,
        mask_threshold=0.8,
        flow_threshold=0.4,
        min_size=15,
        dilation_radius = 12,
        niter=100,
        nclasses=model.nclasses,
        dim=2,
        use_gpu=use_GPU,
    )
    print(f'  masking: {time.time()-tic:.1f}s  |  {masks.max()} cells found')

    results[model_name] = dict(masks=masks, bounds=bounds, flows=flows, iscell=iscell)

# Quick visualisation
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(len(model_names), 3, figsize=(15, 4 * len(model_names)), constrained_layout=True)

for i, model_name in enumerate(model_names):
    r = results[model_name]
    axes[i, 0].imshow(img_arr, cmap='gray')
    if r['masks'].max() > 0:
        axes[i, 0].imshow(r['masks'], cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
    axes[i, 0].set_title(f"{model_name}\nImage + Mask ({r['masks'].max()} cells)")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(r['flows'][0])   # RGB flow
    axes[i, 1].set_title("Flow field")
    axes[i, 1].axis('off')

    fig2_disp = np.expand_dims(r['iscell'],2) * r['flows'][0]
    axes[i, 2].imshow(fig2_disp, cmap='magma')
    axes[i, 2].set_title("RF cell probability mask")
    axes[i, 2].axis('off')

plt.show()

In [ ]:
# Quick visualisation
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(len(model_names), 3, figsize=(15, 4 * len(model_names)), constrained_layout=True)

for i, model_name in enumerate(model_names):
    r = results[model_name]
    axes[i, 0].imshow(img_arr, cmap='gray')
    if r['masks'].max() > 0:
        axes[i, 0].imshow(r['masks'], cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
    axes[i, 0].set_title(f"{model_name}\nImage + Mask ({r['masks'].max()} cells)")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(r['flows'][0])   # RGB flow
    axes[i, 1].set_title("Flow field")
    axes[i, 1].axis('off')

    fig2_disp = np.expand_dims(r['iscell'],2) * r['flows'][0]
    axes[i, 2].imshow(fig2_disp, cmap='magma')
    axes[i, 2].set_title("RF cell probability mask")
    axes[i, 2].axis('off')

plt.show()

In [ ]:
import numpy as np
import omnipose.core
from cellpose_omni import models
from skimage.morphology import disk
from scipy.ndimage import binary_dilation
from scipy.ndimage import binary_fill_holes
import matplotlib.pyplot as plt
import time


def reconstruct_masks_from_flow_magnitude(
    flows,
    flow_mag_threshold=1.0,
    dilation_radius=2,       # disk radius; radius=2 → diameter=5px
    flow_threshold=0.4,
    min_size=15,
    max_size=None,
    niter=None,
    cluster=False,
    nclasses=3,
    dim=2,
    use_gpu=False,
    device=None,
    verbose=False,
    **kwargs
):
    dP = flows[1]
    bd = flows[4]

    if bd is None or (isinstance(bd, np.ndarray) and bd.size == 0):
        bd = np.zeros(dP.shape[1:])

    flow_mag = np.linalg.norm(dP, axis=0)
    iscell = flow_mag > flow_mag_threshold

    if dilation_radius > 0:
        iscell = binary_dilation(iscell, structure=disk(dilation_radius))

    iscell = binary_fill_holes(iscell)
    
    masks, p, tr, bounds, affinity = omnipose.core.compute_masks(
        dP,
        flow_mag,
        bd=bd,
        iscell=iscell,
        niter=niter,
        flow_threshold=flow_threshold,
        min_size=min_size,
        max_size=max_size,
        cluster=cluster,
        nclasses=nclasses,
        dim=dim,
        use_gpu=use_gpu,
        device=device,
        verbose=verbose,
        **kwargs
    )

    return masks, bounds, p, tr, affinity, iscell, flow_mag


# --- Run model ---
model = models.CellposeModel(gpu=use_GPU, model_type='bact_phase_omni')

tic = time.time()
_, flows, _ = model.eval(img_arr, **eval_params)
print(f'network: {time.time()-tic:.1f}s')

# --- Reconstruct masks ---
tic = time.time()
masks, bounds, p, tr, affinity, iscell_dilated, flow_mag = reconstruct_masks_from_flow_magnitude(
    flows=flows,
    flow_mag_threshold=1.0,
    dilation_radius=3,
    niter=200,
    nclasses=model.nclasses,
    dim=model.dim,
    use_gpu=use_GPU,
)
print(f'masking: {time.time()-tic:.1f}s  |  {masks.max()} cells found')

# --- Plot ---
fig, axes = plt.subplots(1, 5, figsize=(25, 5), constrained_layout=True)

axes[0].imshow(img_arr, cmap='gray')
axes[0].set_title('Image')
axes[0].axis('off')

axes[1].imshow(flows[0])
axes[1].set_title('Flow field')
axes[1].axis('off')

axes[2].imshow(iscell_dilated, cmap='gray')
axes[2].set_title('Flow mag > 1 (dilated, d=5px)')
axes[2].axis('off')

axes[3].imshow(img_arr, cmap='gray')
if masks.max() > 0:
    axes[3].imshow(masks, cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
axes[3].set_title(f'Masks ({masks.max()} cells)')
axes[3].axis('off')

axes[4].imshow(flow_mag, cmap='viridis')
axes[4].set_title('Flow magnitude')
axes[4].axis('off')

plt.show()

In [ ]:
# --- Plot ---
fig, axes = plt.subplots(1, 5, figsize=(25, 5), constrained_layout=True)

axes[0].imshow(img_arr, cmap='gray')
axes[0].set_title('Image')
axes[0].axis('off')

axes[1].imshow(flows[0])
axes[1].set_title('Flow field')
axes[1].axis('off')

axes[2].imshow(iscell_dilated, cmap='gray')
axes[2].set_title('Flow mag > 1 (dilated, d=5px)')
axes[2].axis('off')

axes[3].imshow(img_arr, cmap='gray')
if masks.max() > 0:
    axes[3].imshow(masks, cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
axes[3].set_title(f'Masks ({masks.max()} cells)')
axes[3].axis('off')

axes[4].imshow(flow_mag, cmap='viridis')
axes[4].set_title('Flow magnitude')
axes[4].axis('off')

plt.show()

## 3d omnipose

Requires 20GB of RAM or VRAM. Cannot run the model on my pc.

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer

search_path = (
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET145/01",
)

file_pattern = ("CET*DAPI*.tif",
                "CET*Cy5*.tif",
                "*DIC*.tif",
                "MAX_CET145*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0,
        "quantization": "8bit",
        "correct_DIC_shift" : [5,22], # [Y,X] more negative Y to shift up, more positive X to shift to right
        "two_channel_merge_mode": "passthrough"
    },
    
}

dapi_files = []
cy5_files = []
for p in search_path: 
    dapi_files.append(find_files_by_pattern(p, file_pattern[0], verbose=True))
    cy5_files.append(find_files_by_pattern(p, file_pattern[1], verbose=True))

In [ ]:
dapi_c = ImageContainer(dapi_files[0],config)
cy5_c = ImageContainer(cy5_files[0],config)
merged_c = cy5_c + dapi_c
merged_img = merged_c.merge()
merged_img.shape

In [ ]:
imgs = [merged_img]

In [ ]:
# Import dependencies
import numpy as np
from cellpose_omni import models, core

# This checks to see if you have set up your GPU properly.
# CPU performance is a lot slower, but not a problem if you 
# are only processing a few images.
use_GPU = core.use_gpu()

# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['figure.dpi'] = 300
plt.style.use('dark_background')
%matplotlib inline

In [ ]:
from cellpose_omni import models
model_name = 'plant_omni'

dim = 3
nclasses = 3 # flow + dist + boundary
nchan = 1
omni = 1
rescale = False
diam_mean = 0
use_GPU = 0 # Most people do not have enough VRAM to run on GPU... 24GB not enough for this image, need nearly 48GB
model = models.CellposeModel(gpu=use_GPU, model_type=model_name, net_avg=False, 
                             diam_mean=diam_mean, nclasses=nclasses, dim=dim, nchan=nchan)

In [ ]:
import torch
torch.cuda.empty_cache()
mask_threshold = -5 #usually this is -1
flow_threshold = 0.
diam_threshold = 12
net_avg = False
cluster = False
verbose = 1
tile = 0
chans = None
compute_masks = 1
resample=False
rescale=None
omni=True
flow_factor = 10 # multiple to increase flow magnitude, useful in 3D
transparency = True

nimg = len(imgs)
masks_om, flows_om = [[]]*nimg,[[]]*nimg

# splitting the images into batches helps manage VRAM use so that memory can get properly released 
# here we have just one image, but most people will have several to process
for k in range(nimg):
    masks_om[k], flows_om[k], _ = model.eval(imgs[k],
                                             channels=chans,
                                             rescale=rescale,
                                             mask_threshold=mask_threshold,
                                             net_avg=net_avg,
                                             transparency=transparency,
                                             flow_threshold=flow_threshold,
                                             omni=omni,
                                             resample=resample,
                                             verbose=verbose,
                                             diam_threshold=diam_threshold,
                                             cluster=cluster,
                                             niter=10,
                                             tile=tile,
                                             compute_masks=compute_masks,
                                             flow_factor=flow_factor) 